# Step 42 — Exact source-index comparison

Read the separate step-40 and selected step-41 CSVs; do not merge origins or alter inputs. Exact equality is tested on origin-time **strings**, latitude, longitude, depth_km and magnitude after dropping duplicate five-field rows within each source. This is intentionally stricter than a physical-event match: timestamp formatting, rounding, solution revisions, different coverage and clock conventions can all produce nonmatches.

Step 40 writes a reported magnitude of 0.00 (no magnitude computed) as a missing magnitude; `reported_magnitude` keeps the 0.00. Two rows with missing magnitudes match each other here, so a located origin with and without a magnitude counts as a mismatch only when one source has a real magnitude and the other does not. Pandas merge semantics match missing values in join keys; an exact match does not guarantee complete metadata. Source provenance and quality fields are not compared. The two output difference CSVs contain the five comparison fields, not full provenance; look back to the source indexes/raw lines to investigate.

`out_prefix` selects the output parent directory; filenames remain fixed as `42_pinaall_not_in_40.csv` and `42_40_not_in_pinaall.csv`. Outputs overwrite existing files. For annotated PINAALL, explicitly select its separate CSV and resolve time policies first. Step 43 provides a tolerant comparison, not a reason to force exact agreement.

## 1. Configuration

Review source paths, selected PINAALL variant and output paths before running. Configuration does not execute the workflow.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

# Find the adjacent local package whether Jupyter starts here or in the project root.
_candidates = [Path.cwd(), Path.cwd() / '10_conversion', *Path.cwd().parents]
PIPELINE_DIR = next((p for p in _candidates if (p / 'lib' / 'clock.py').is_file()), None)
if PIPELINE_DIR is None:
    raise FileNotFoundError('Open this notebook from the pipeline folder or project root.')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
from lib.config import pipeline_paths

CONFIG_FILE = None  # Default: conversion_config.json; or set an explicit JSON path.
P = pipeline_paths(config_file=CONFIG_FILE)
DATA_TOP = Path(P['DATA_TOP'])
print('Configuration:', P['CONFIG_FILE'])
print('External outputs:', P['OUTPUT_ROOT'])

SETTINGS = dict(
    hypo40=P['SUMMARY_FILE_40_INDEX'],
    hypo41=P['SUMMARY_FILE_41_INDEX'],
    out_prefix=P['COMPARE_PREFIX'],
)
display(SETTINGS)

## 2. Implementation

The complete step-specific workflow is defined below. Definitions alone do not read sources or write products.

In [ ]:
from pathlib import Path
import pandas as pd
from lib.shared import mark_missing_magnitudes

COMPARE_COLS = [
    "origin_time",
    "latitude",
    "longitude",
    "depth_km",
    "magnitude",
]

def run(
    *,
    hypo40,
    hypo41,
    out_prefix
):
    """Run this Pinatubo step with explicit notebook arguments. Return tables and results for inspection."""
    from types import SimpleNamespace
    args = SimpleNamespace(
        hypo40=hypo40,
        hypo41=hypo41,
        out_prefix=out_prefix,
    )

    # Indexes from before the magnitude fix still hold 0.00; treat it as missing here too.
    df40 = mark_missing_magnitudes(pd.read_csv(args.hypo40))[COMPARE_COLS].drop_duplicates()
    df41 = mark_missing_magnitudes(pd.read_csv(args.hypo41))[COMPARE_COLS].drop_duplicates()

    merged = df41.merge(df40, how="left", on=COMPARE_COLS, indicator=True)
    pinaall_not_in_40 = merged[merged["_merge"] == "left_only"].drop(columns="_merge")

    merged_rev = df40.merge(df41, how="left", on=COMPARE_COLS, indicator=True)
    hypo40_not_in_pinaall = merged_rev[merged_rev["_merge"] == "left_only"].drop(columns="_merge")

    out_dir = Path(args.out_prefix).parent
    out_dir.mkdir(parents=True, exist_ok=True)

    out1 = out_dir / "42_pinaall_not_in_40.csv"
    out2 = out_dir / "42_40_not_in_pinaall.csv"

    pinaall_not_in_40.to_csv(out1, index=False)
    hypo40_not_in_pinaall.to_csv(out2, index=False)

    print("\nHYPOCENTER INDEX COMPARISON (EXACT MATCH)")
    print("========================================")
    print(f"Rows in 40: {len(df40)}")
    print(f"Rows in 41: {len(df41)}\n")
    print(f"PINAALL rows found in 40: {len(df41) - len(pinaall_not_in_40)}")
    print(f"PINAALL rows NOT in 40:   {len(pinaall_not_in_40)}\n")
    print(f"40 rows NOT in PINAALL:   {len(hypo40_not_in_pinaall)}\n")
    print(f"Wrote: {out1}")
    print(f"Wrote: {out2}")
    return {k: v for k, v in locals().items() if (k.startswith("df") or k.startswith("catalog") or k.endswith("_rows") or k in ("events", "annotations", "errors", "stats", "diag", "merged", "suppressed", "matches", "rows", "unparsed"))}

## 3. Execute when ready

This cell reads configured inputs and replaces configured outputs. Inspect summaries and diagnostic tables after the run.

In [ ]:
result = run(**SETTINGS)

In [ ]:
# Preview returned tables and summary objects without reloading output files.
for name, value in result.items():
    print(name)
    display(value.head() if hasattr(value, 'head') else
            value[:5] if isinstance(value, list) else value)
